In [1]:
from agent_prompts.base_agent_prompt import get_function_calls

MODEL = "btrabucco/Insta-Qwen2.5-1.5B-GRPO-n1"
FINETUNED_REPO = "saadashraf87/insta-qwen-2.5-1.7b-grpo-on-policy"
OUTPUT_DIR = "/models/insta-qwen-2.5-1.7b-grpo-on-policy"
HF_TOKEN = None
DATASET_NAME = "data-for-agents/insta-150k-v3"
PRECISION = "fp16"
SAFE_SERIALIZATION = True

USE_LORA = True
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_MERGE_AT_SAVE = True

SFT_WARMUP = True
SFT_EPOCHS = 1
SFT_BATCH_SIZE = 4
SFT_LR = 2e-4

PPO_STEPS = 2000
PPO_BATCH_SIZE = 64
PPO_FORWARD_BATCH_SIZE = 4
PPO_EPOCHS = 4
PPO_LR = 1.41e-5
PPO_CLIP_RANGE = 0.2
KL_COEFF = 0.02
GAMMA = 1.0
LAMBDA = 0.95
ROLLOUT_MAXLEN = 512
SAVE_EVERY_N_UPDATES = 1

REWARD_MODEL = None

OUTPUT_INTERMEDIATE_CHECKPOINTS = True

PLAYWRIGHT_SERVER_URL = "http://localhost:3000"


In [11]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from server_client import ServerClient
from utils import is_stop_action
import torch

In [12]:
dataset = load_dataset(DATASET_NAME)

In [13]:
print(dataset['train'][:5])

{'website': ['railway.gov.tw', 'gameandfishmag.com', 'iflysouthern.com', 'tut.fi', 'visit-hannover.com'], 'instruction': ['Find the earliest train from Taipei to Kaohsiung on 2025.05.20 and state its departure time and fare for a standard adult ticket.', 'Find an article on the website that discusses spring bass fishing in Wisconsin and identify at least two specific lakes or rivers mentioned in that article.', 'Find out if there are any direct flights from Dallas (Dallas/Fort Worth International Airport - DFW) to Hot Springs, Arkansas (Hot Springs Memorial Field Airport - HOT) for a trip departing on June 15th and returning on June 20th of the current year. If so, what is the earliest departure time available for the outbound flight on June 15th?', "Find the contact information for the 'Faculty of Information Technology and Communication Sciences' at Tampere University.", 'Find the general opening hours for the New Town Hall (Neues Rathaus) on the official Hannover tourism website.'],

In [14]:
train_dataset = dataset['train']

In [16]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL)

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 0967da7b-14eb-48fb-8103-e236b0cb7bd3)')' thrown while requesting HEAD https://huggingface.co/btrabucco/Insta-Qwen2.5-1.5B-GRPO-n1/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


  2025-09-30T19:14:54.013957Z  WARN  Reqwest(reqwest::Error { kind: Request, url: "https://transfer.xethub.hf.co/xorbs/default/8f0fe24ae70ba018b5abe266f20944ee7859e2c0c417870df367bc3f24b3b03b?X-Xet-Signed-Range=bytes%3D0-58520876&X-Xet-Session-Id=01K6DYTGYDSQ574YHC69MKBXD3&Expires=1759261250&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly90cmFuc2Zlci54ZXRodWIuaGYuY28veG9yYnMvZGVmYXVsdC84ZjBmZTI0YWU3MGJhMDE4YjVhYmUyNjZmMjA5NDRlZTc4NTllMmMwYzQxNzg3MGRmMzY3YmMzZjI0YjNiMDNiP1gtWGV0LVNpZ25lZC1SYW5nZT1ieXRlcyUzRDAtNTg1MjA4NzYmWC1YZXQtU2Vzc2lvbi1JZD0wMUs2RFlUR1lEU1E1NzRZSEM2OU1LQlhEMyIsIkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc1OTI2MTI1MH19fV19&Signature=ZocuPP8GUdAE9zCR6DKLX8h43Rp66xtp1Ar2IQqIhB2jbuAsuk0CG~XjlI3hxhiLN0zH6gUbl2-IV-PorIFDalTibSLl9w8bUtWcLS8mCmJcPu9O3dG8lwegOe4zFfelxa0f6YMwvj0b9kfb7uNxUDpqUU6POPWEQp8U7cPuo5eNa9I0MdmbDwXdOHyD~LYTDcVd9JtKjqWIxp3FOD23D8uAqBu6XA5dkQ7Waj9zr537Gig3-hM7um1ShByPuySeR2DCwrKWP27kOXB~cxaT9xRGvxxfqhrxnJoZv9JH7v6YhTvhMH~rkRN69Gwfh

OSError: Can't load the model for 'btrabucco/Insta-Qwen2.5-1.5B-GRPO-n1'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'btrabucco/Insta-Qwen2.5-1.5B-GRPO-n1' is the correct path to a directory containing a file named pytorch_model.bin, tf_model.h5, model.ckpt or flax_model.msgpack.

In [ ]:
def generate(prompt, max_new_tokens=128, temperature=0.8, top_k=50, top_p=0.95):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        gen = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id
        )
    text = tokenizer.decode(gen[0], skip_special_tokens=True)
    return text

In [12]:
websites = train_dataset['website']
instructions = train_dataset['instruction']
steps = train_dataset['steps']
criteria = train_dataset['criteria']

In [ ]:
# Start a Playwright session once at the beginning using ServerClient
client = ServerClient(server_url=PLAYWRIGHT_SERVER_URL)
session_id = client.start_session(width=1920, height=1080)
print(f"Started Playwright session: {session_id}")

try:
    for index, website in enumerate(websites):
        instruction = instructions[index]
        print(f"\nProcessing website {index + 1}/{len(websites)}: {website}")
        
        client.navigate(website)
        client.get_observation()
        
        model_response = ''
        action = {}
        while not is_stop_action(action):
            markdown = client.get_observation(session_id)
            model_response = generate(markdown)
            parsed_action_json = {}
            action = get_function_calls(parsed_action_json)
            client.execute_action(action)
            
finally:
    # Close the session when done
    client.close_session()
    print(f"Closed Playwright session: {session_id}")

railway.gov.tw
gameandfishmag.com
iflysouthern.com
tut.fi
visit-hannover.com
britishecologicalsociety.org
biggestmorningtea.com.au
calculatorcat.com
ulc.ca
visitmoretonbayregion.com.au
luxurylink.com
littletikes.com
aquashowpark.com
ssvpusa.org
millsdentalgroup.com
chem-eng.utoronto.ca
recyclenation.com
statutes.capitol.texas.gov
smitegame.com
londonnorthwesternrailway.co.uk
literaturuebersetzer.de
tenders.gov.au
marx.ruc.edu.cn
freefavicon.com
professional.diabetes.org
hautesavoie.fr
franchisebusinessreview.com
coretrustseal.org
checkyourfact.com
health.costhelper.com
nnlm.gov
childrensmiraclenetworkhospitals.org
astro.unl.edu
helpcenter.steinberg.de
123-parking.co.uk
simpleprogrammer.com
manual.collectiveaccess.org
peterboroughpublichealth.ca
mdah.ms.gov
easternct.edu
psb.ugent.be
woodgreen.org.uk
minidsp.com
matthewjamestaylor.com
milwaukeeindependent.com
swarovskioptik.com
support.apple.com
alabamaworks.alabama.gov
cyberstates.org
genomenewsnetwork.org
thesiswhisperer.com
novedge.c